# CNB OAM Document Classifier — RobeCzech Fine-Tuning on Google Colab

This notebook contains the complete pipeline to load your OAM SQLite database (`metadata.db`), preprocess texts, and fine-tune a pre-trained Czech language model (**RobeCzech**) on a Google Colab GPU.

### Instructions:
1. Open this notebook in [Google Colab](https://colab.research.google.com/).
2. In Colab, select **Runtime -> Change runtime type** and choose **T4 GPU** (or any other available GPU accelerator).
3. Upload your local SQLite database `metadata.db` to the Colab environment (click the folder icon in the left sidebar and drag-and-drop the file).
4. Run the cells sequentially.

## 1. Setup & Installation

In [ ]:
# Install HuggingFace libraries, scikit-learn, and visualizers
!pip install -q transformers[torch] datasets accelerate scikit-learn pandas matplotlib seaborn

## 2. Load Dataset from SQLite

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

db_path = "metadata.db" # Ensure metadata.db is uploaded in the root Colab directory

# Categories list from configurations
categories = [
    "Oznámení o konání valné hromady",
    "Informace související s valnou hromadou",
    "Informace související s emisí dluhopisů",
    "Výroční finanční zpráva",
    "Pololetní finanční zpráva",
    "Vnitřní informace",
    "Oznámení podílu na hlasovacích právech",
    "Informace o celkovém počtu hlasovacích práv a výši základního kapitálu",
    "Oznámení o konání schůze vlastníků",
    "Informace o nabytí nebo pozbytí vlastních akcií emitenta",
    "Zpráva o úhradách placených státu",
    "Samostatná zpráva o nefinančních informacích"
]

# Query database to load document ID, type and text
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
query = """
    SELECT 
        d.id, 
        d.typ_informace, 
        GROUP_CONCAT(et.text_content, '\n\n') as full_text
    FROM documents d
    JOIN extracted_text et ON d.id = et.document_id
    GROUP BY d.id
"""

texts = []
labels = []

cursor = conn.execute(query)
for row in cursor.fetchall():
    lbl = row["typ_informace"]
    txt = row["full_text"]
    if lbl in categories and txt and len(txt.strip()) > 50:
        texts.append(txt.strip())
        labels.append(lbl)
conn.close()

print(f"Successfully loaded {len(texts)} Czech documents.")

# Check class distribution
df_dist = pd.Series(labels).value_counts().to_frame("counts")
print("\nClass distribution:")
print(df_dist)

## 3. Preprocess and Split Data

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified train/test split (80% train, 20% test)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

unique_labels = sorted(list(set(labels)))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(unique_labels)

print(f"Training set size: {len(train_texts)}")
print(f"Test set size: {len(test_texts)}")
print(f"Number of classes: {num_labels}")

## 4. Fine-Tuning Setup with RobeCzech

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight

model_name = "ufal/robeczech-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load datasets
train_dataset = Dataset.from_dict({"text": train_texts, "label": [label2id[l] for l in train_labels]})
test_dataset = Dataset.from_dict({"text": test_texts, "label": [label2id[l] for l in test_labels]})

def tokenize_func(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

tokenized_train = train_dataset.map(tokenize_func, batched=True)
tokenized_test = test_dataset.map(tokenize_func, batched=True)

# Compute class weights to tackle imbalance
numeric_train_labels = [label2id[l] for l in train_labels]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_labels),
    y=numeric_train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# Weighted Cross Entropy Loss Trainer
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Evaluation metrics call
def compute_metrics(eval_pred):
    from sklearn.metrics import accuracy_score, f1_score
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "f1": f1}

# Load pre-trained RobeCzech model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label
)

## 5. Train Model

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

# Run training
trainer.train()

## 6. Evaluate & Plot Visualizations

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Get predictions
preds_output = trainer.predict(tokenized_test)
logits = preds_output.predictions
predictions = np.argmax(logits, axis=1)
y_true = [label2id[l] for l in test_labels]

# Print classification report
print(classification_report(y_true, predictions, target_names=unique_labels))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=unique_labels, yticklabels=unique_labels)
plt.title("Confusion Matrix - RobeCzech")
plt.ylabel("True Category")
plt.xlabel("Predicted Category")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Export Model for Local Inference

In [ ]:
import json
from pathlib import Path

# Save model
save_path = Path("./models/bert")
save_path.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Write mappings
mappings = {
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "model_name": model_name,
}
with open(save_path / "mappings.json", "w", encoding="utf-8") as f:
    json.dump(mappings, f, ensure_ascii=False, indent=2)

# Zip the model and files for easy download
!zip -r robeczech_finetuned.zip models/
print("\n--- Model saved and packaged! ---")
print("Please download 'robeczech_finetuned.zip' from Colab's file explorer and extract it in your local project root as 'models/bert'.")